In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../output/support.db")

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path

DATA = Path("../data/processedCombined_Data.xlsx")
DB = Path("../output/support.db")

engine = create_engine(f"sqlite:///{DB}")

excel = pd.ExcelFile(DATA)

print(excel.sheet_names)

for sheet in excel.sheet_names:
    df = pd.read_excel(DATA, sheet_name=sheet)
    table = sheet.replace(" ", "_")
    df.to_sql(table, engine, if_exists="replace", index=False)
    print(f"Loaded {table}")

print("\nDatabase Created Successfully")

In [ ]:
from pathlib import Path

def load_query(name):
    with open(Path("../queries") / f"{name}.sql", "r") as f:
        return f.read()

In [ ]:
monthly = pd.read_sql(
    load_query("monthly_ticket_summary"),
    engine
)

segment = pd.read_sql(
    load_query("ticket_segment_analysis"),
    engine
)

funnel = pd.read_sql(
    load_query("support_funnel"),
    engine
)

print(monthly.head())
print(segment.head())
print(funnel.head())

In [ ]:
def validate_metrics(monthly, segment, funnel):

    assert monthly.isnull().sum().sum() == 0
    assert segment.isnull().sum().sum() == 0
    assert funnel.isnull().sum().sum() == 0

    assert (segment["total_tickets"] >= 0).all()
    assert (segment["closed_tickets"] >= 0).all()

    assert (funnel["closure_percentage"] >= 0).all()
    assert (funnel["closure_percentage"] <= 100).all()

    print("✓ Validation Successful")

validate_metrics(monthly, segment, funnel)